## Example 1: C extension components

In [ ]:
# Example 1: C extension components
from _ruamel_yaml import CParser, CEmitter

print("Testing ruamel.yaml.clib C extension:")

# Check C extension classes
print(f"✓ CParser: {CParser}")
print(f"✓ CEmitter: {CEmitter}")

# Check if they're C extension types
print(f"✓ CParser type: {type(CParser).__name__}")
print(f"✓ CEmitter type: {type(CEmitter).__name__}")

## Example 2: Integration with ruamel.yaml

In [ ]:
# Example 2: Integration with ruamel.yaml
# The C extension is used through ruamel.yaml, which ruamel.yaml.clib does not install: pip install ruamel.yaml
from ruamel.yaml import YAML
from _ruamel_yaml import CParser, CEmitter
import io

print("Testing ruamel.yaml with C extension:")

# Only YAML(typ='safe') uses the C parser and emitter; the default round-trip YAML() is pure Python.
yaml = YAML(typ='safe')
assert yaml.Parser is CParser and yaml.Emitter is CEmitter, (yaml.Parser, yaml.Emitter)

# Test data
data = {'key': 'value', 'list': [1, 2, 3], 'nested': {'a': 1, 'b': 2}}

# Serialize with the C emitter
stream = io.StringIO()
yaml.dump(data, stream)
yaml_str = stream.getvalue()
print(f"✓ Serialized: {len(yaml_str)} chars")

# Deserialize with the C parser
loaded = yaml.load(yaml_str)
assert loaded == data, loaded
print(f"✓ Loaded: {loaded}")

# The pure-Python safe loader reads the same document to the same data
pure = YAML(typ='safe', pure=True)
assert pure.Parser is not CParser, pure.Parser
assert pure.load(yaml_str) == loaded
print("✓ C and pure-Python loaders agree")

## Example 3: C extension module inspection

In [ ]:
# Example 3: C extension module inspection
import _ruamel_yaml

print("Testing C extension module:")

# List all attributes
attrs = [a for a in dir(_ruamel_yaml) if not a.startswith('_')]
print(f"✓ Public attributes: {len(attrs)}")
print(f"✓ Sample: {attrs[:5] if len(attrs) >= 5 else attrs}")

# Check module type
print(f"✓ Module type: {type(_ruamel_yaml).__name__}")

# Check for main classes
main_classes = ['CParser', 'CEmitter']
for cls_name in main_classes:
    if hasattr(_ruamel_yaml, cls_name):
        cls = getattr(_ruamel_yaml, cls_name)
        print(f"✓ {cls_name}: available")

## Example 4: C and pure-Python safe loaders read the same data

In [ ]:
# Example 4: C and pure-Python safe loaders read the same data
# The C extension is used through ruamel.yaml, which ruamel.yaml.clib does not install: pip install ruamel.yaml
from ruamel.yaml import YAML
from _ruamel_yaml import CParser
import io
import time

print("Comparing the C and pure-Python safe loaders:")

c_yaml = YAML(typ='safe')                # C parser and emitter from ruamel.yaml.clib
pure_yaml = YAML(typ='safe', pure=True)  # pure-Python parser and emitter
assert c_yaml.Parser is CParser, c_yaml.Parser
assert pure_yaml.Parser is not CParser, pure_yaml.Parser

# Generate test data and serialize it with the C emitter
data = {f'key_{i}': {f'sub_{j}': j for j in range(10)} for i in range(50)}
stream = io.StringIO()
c_yaml.dump(data, stream)
document = stream.getvalue()

timings = {}
for label, loader in (('C', c_yaml), ('pure Python', pure_yaml)):
    start = time.perf_counter()
    for _ in range(10):
        loaded = loader.load(document)
    timings[label] = time.perf_counter() - start
    assert loaded == data, label

print(f"✓ Test data: {len(data)} top-level keys, {len(document)} chars of YAML")
# The timings are for information only: they depend on the machine.
for label, seconds in timings.items():
    print(f"✓ 10 loads with the {label} loader: {seconds:.4f}s")

## Example 5: C extension availability check

In [ ]:
# Example 5: C extension availability check
# The C extension is used through ruamel.yaml, which ruamel.yaml.clib does not install: pip install ruamel.yaml
print("Testing C extension availability:")

# Check if C extension is available
# _ruamel_yaml is the compiled extension this package ships (ruamel.yaml.clib has nothing else): import it directly
from _ruamel_yaml import CParser, CEmitter
print(f"✓ C extension available")
print(f"✓ CParser: {CParser.__name__}")
print(f"✓ CEmitter: {CEmitter.__name__}")

# Check integration with ruamel.yaml: the 'safe' loader and dumper use the C classes when the extension
# imports; the default round-trip YAML() is pure Python
from ruamel.yaml import YAML
yaml = YAML(typ="safe")
assert yaml.Parser is CParser and yaml.Emitter is CEmitter, (yaml.Parser, yaml.Emitter)

# Test with simple YAML
test_str = "key: value\nlist:\n  - item1\n  - item2"
data = yaml.load(test_str)
assert data == {"key": "value", "list": ["item1", "item2"]}, data
print(f"✓ YAML loaded: {data['key']}")
print(f"✓ Using C extension: {yaml.Parser is CParser}")